# Silver Layer

This notebook cleans and prepares the raw data.
- fix date formats
- clean sensor status
- remove invalid records
- remove duplicates
- create joined parcel time-series data

In [0]:
#Import Libraries

import pyspark.sql.functions as F
from pyspark.sql.types import StringType, IntegerType, DateType, TimestampType, FloatType



In [0]:
# Define Catalog Name
catalog_name = 'carnot-catalog'

In [0]:
#Read metadata_bronze File


metadata_bronze = spark.table(f"`{catalog_name}`.bronze.bronze_parcel_metadata")

display(metadata_bronze.limit(5))

parcel_id,mill_id,crop_type,sowing_date,area_hectares,_source_file,ingested_at
PARCEL_001,MILL_NORTH,sugarcane,2026-02-10,9.03,dbfs:/Volumes/carnot-catalog/source-data/rawdata/parcel_metadata.csv,2026-05-18T13:04:14.605Z
PARCEL_002,MILL_SOUTH,sugarcane,2026-01-16,8.97,dbfs:/Volumes/carnot-catalog/source-data/rawdata/parcel_metadata.csv,2026-05-18T13:04:14.605Z
PARCEL_003,MILL_NORTH,soybean,2026-02-13,5.35,dbfs:/Volumes/carnot-catalog/source-data/rawdata/parcel_metadata.csv,2026-05-18T13:04:14.605Z
PARCEL_004,MILL_NORTH,sugarcane,2026-01-02,3.18,dbfs:/Volumes/carnot-catalog/source-data/rawdata/parcel_metadata.csv,2026-05-18T13:04:14.605Z
PARCEL_005,MILL_NORTH,soybean,2026-02-08,2.79,dbfs:/Volumes/carnot-catalog/source-data/rawdata/parcel_metadata.csv,2026-05-18T13:04:14.605Z


In [0]:
#Read Parcel Readings Bronze File

parcel_readings_bronze = spark.table(f"`{catalog_name}`.bronze.bronze_parcel_readings")

display(parcel_readings_bronze.limit(5))

parcel_id,date,ndvi_value,temperature_c,rainfall_mm,sensor_status,_source_file,ingested_at
PARCEL_018,16/05/2026,0.595,15.4,0.0,error,dbfs:/Volumes/carnot-catalog/source-data/rawdata/parcel_readings.csv,2026-05-18T13:04:35.677Z
PARCEL_014,2026-01-27,0.457,27.6,0.0,OK,dbfs:/Volumes/carnot-catalog/source-data/rawdata/parcel_readings.csv,2026-05-18T13:04:35.677Z
PARCEL_006,2026-05-17,0.497,25.8,0.0,OK,dbfs:/Volumes/carnot-catalog/source-data/rawdata/parcel_readings.csv,2026-05-18T13:04:35.677Z
PARCEL_004,10/02/2026,0.81,25.0,0.0,OK,dbfs:/Volumes/carnot-catalog/source-data/rawdata/parcel_readings.csv,2026-05-18T13:04:35.677Z
PARCEL_002,2026-01-17,0.269,33.2,0.0,OK,dbfs:/Volumes/carnot-catalog/source-data/rawdata/parcel_readings.csv,2026-05-18T13:04:35.677Z


In [0]:
#Check Row Count for Readings and Metadata Files
readings_total = parcel_readings_bronze.count()
metadata_total = metadata_bronze.count()

print("Readings:", readings_total)
print("Metadata:", metadata_total)

Readings: 3447
Metadata: 28


In [0]:
#Data Cleaning For parcel Readings

Data Cleaning For parcel_readings

### Date format check

The readings file has mixed date formats, so this needs validation before parsing

In [0]:
#Validate and Fixed Date format for readings_df

parcel_readings_silver = parcel_readings_bronze.withColumn(
    "reading_date",
    F.coalesce(
        F.try_to_date(F.col("date"), "yyyy-MM-dd"),
        F.try_to_date(F.col("date"), "dd/MM/yyyy"),
        F.try_to_date(F.col("date"), "dd-MMM-yyyy")
    )
)

In [0]:
#Removed  Invalid Dates 

parcel_readings_silver = parcel_readings_silver.filter(
    F.col("reading_date").isNotNull()
)

In [0]:
display (parcel_readings_silver.limit(5))

parcel_id,date,ndvi_value,temperature_c,rainfall_mm,sensor_status,_source_file,ingested_at,reading_date
PARCEL_018,16/05/2026,0.595,15.4,0.0,error,dbfs:/Volumes/carnot-catalog/source-data/rawdata/parcel_readings.csv,2026-05-18T13:04:35.677Z,2026-05-16
PARCEL_014,2026-01-27,0.457,27.6,0.0,OK,dbfs:/Volumes/carnot-catalog/source-data/rawdata/parcel_readings.csv,2026-05-18T13:04:35.677Z,2026-01-27
PARCEL_006,2026-05-17,0.497,25.8,0.0,OK,dbfs:/Volumes/carnot-catalog/source-data/rawdata/parcel_readings.csv,2026-05-18T13:04:35.677Z,2026-05-17
PARCEL_004,10/02/2026,0.81,25.0,0.0,OK,dbfs:/Volumes/carnot-catalog/source-data/rawdata/parcel_readings.csv,2026-05-18T13:04:35.677Z,2026-02-10
PARCEL_002,2026-01-17,0.269,33.2,0.0,OK,dbfs:/Volumes/carnot-catalog/source-data/rawdata/parcel_readings.csv,2026-05-18T13:04:35.677Z,2026-01-17


In [0]:
#Normalize Sensor Status

parcel_readings_silver = parcel_readings_silver.withColumn(
    "sensor_status",
    F.lower(F.trim(F.col("sensor_status")))
)

In [0]:
#Remove Invalid Ndvi

parcel_readings_silver = parcel_readings_silver.filter(
    (F.col("ndvi_value").isNull()) |
    (
        (F.col("ndvi_value") >= -1) &
        (F.col("ndvi_value") <= 1)
    )
)

In [0]:

# we need one reading per day for parcel reading
# Remove Duplicate Records


parcel_readings_silver = parcel_readings_silver.dropDuplicates(
    ["parcel_id", "reading_date"]
)

In [0]:
display(parcel_readings_silver.limit(5))

parcel_id,date,ndvi_value,temperature_c,rainfall_mm,sensor_status,_source_file,ingested_at,reading_date
PARCEL_018,16/05/2026,0.595,15.4,0.0,error,dbfs:/Volumes/carnot-catalog/source-data/rawdata/parcel_readings.csv,2026-05-18T13:04:35.677Z,2026-05-16
PARCEL_014,2026-01-27,0.457,27.6,0.0,ok,dbfs:/Volumes/carnot-catalog/source-data/rawdata/parcel_readings.csv,2026-05-18T13:04:35.677Z,2026-01-27
PARCEL_006,2026-05-17,0.497,25.8,0.0,ok,dbfs:/Volumes/carnot-catalog/source-data/rawdata/parcel_readings.csv,2026-05-18T13:04:35.677Z,2026-05-17
PARCEL_004,10/02/2026,0.81,25.0,0.0,ok,dbfs:/Volumes/carnot-catalog/source-data/rawdata/parcel_readings.csv,2026-05-18T13:04:35.677Z,2026-02-10
PARCEL_002,2026-01-17,0.269,33.2,0.0,ok,dbfs:/Volumes/carnot-catalog/source-data/rawdata/parcel_readings.csv,2026-05-18T13:04:35.677Z,2026-01-17


In [0]:
# Save cleaned parcel readings to Silver


parcel_readings_silver.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        f"`{catalog_name}`.silver.silver_parcel_readings"
    )


Data Cleaning for Metadata 

In [0]:
#Fixed date issue for Metadata

metadata_silver = metadata_bronze.withColumn(
    "sowing_date",
    F.try_to_date(F.col("sowing_date"), "yyyy-MM-dd")
)

In [0]:
#Remove Invalid Area

metadata_silver = metadata_silver.filter(
    F.col("area_hectares") > 0
)

In [0]:

#Save Metadata Silver file

metadata_silver.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        f"`{catalog_name}`.silver.silver_parcel_metadata"
    )

In [0]:
# Create conformed parcel time-series dataset



parcel_timeseries_silver = parcel_readings_silver.join(
    metadata_silver,
    "parcel_id",
    "inner"
)

In [0]:
#select Columns for timeseries

parcel_timeseries_silver = parcel_timeseries_silver.select(
    "parcel_id",
    "reading_date",
    "ndvi_value",
    "temperature_c",
    "rainfall_mm",
    "sensor_status",
    "mill_id",
    "crop_type",
    "sowing_date",
    "area_hectares"
)

In [0]:
display(parcel_timeseries_silver.limit(5))

parcel_id,reading_date,ndvi_value,temperature_c,rainfall_mm,sensor_status,mill_id,crop_type,sowing_date,area_hectares
PARCEL_015,2026-01-11,0.257,34.0,0.0,ok,MILL_SOUTH,sugarcane,2026-01-06,4.87
PARCEL_014,2026-03-18,0.769,18.0,7.8,ok,MILL_NORTH,sugarcane,2026-01-05,9.39
PARCEL_014,2026-01-07,0.288,19.4,0.0,ok,MILL_NORTH,sugarcane,2026-01-05,9.39
PARCEL_015,2026-02-13,0.802,16.3,0.0,ok,MILL_SOUTH,sugarcane,2026-01-06,4.87
PARCEL_019,2026-01-30,0.335,24.7,0.0,ok,MILL_SOUTH,sugarcane,2026-01-18,10.19


In [0]:
#save timeseries to silver table 

parcel_timeseries_silver.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        f"`{catalog_name}`.silver.silver_parcel_timeseries"
    )

In [0]:
# save output in csv

parcel_timeseries_silver.write.mode("overwrite").option("header", True).csv(f"/Volumes/{catalog_name}/silver/csv_exports/cleaned_parcel_timeseries_csv")

In [0]:
# Create Audit table based on records
import builtins

# invalid date records
invalid_dates = parcel_readings_bronze.filter(
    F.coalesce(
        F.try_to_date(F.col("date"), "yyyy-MM-dd"),
        F.try_to_date(F.col("date"), "dd/MM/yyyy"),
        F.try_to_date(F.col("date"), "dd-MMM-yyyy")
    ).isNull()
).count()

# invalid NDVI
invalid_ndvi = parcel_readings_bronze.filter(
    (F.col("ndvi_value") < -1) |
    (F.col("ndvi_value") > 1)
).count()

# duplicate parcel-date
duplicate_count = parcel_readings_bronze.withColumn(
    "parsed_date",
    F.coalesce(
        F.try_to_date(F.col("date"), "yyyy-MM-dd"),
        F.try_to_date(F.col("date"), "dd/MM/yyyy"),
        F.try_to_date(F.col("date"), "dd-MMM-yyyy")
    )
).groupBy(
    "parcel_id",
    "parsed_date"
).count().filter(
    F.col("count") > 1
).count()

# bad sensor
sensor_issues = parcel_readings_bronze.filter(
    F.lower(F.trim(F.col("sensor_status"))) != "ok"
).count()

# orphan parcel ids
orphan_parcels = parcel_readings_bronze.select("parcel_id").distinct().join(
    metadata_bronze.select("parcel_id").distinct(),
    "parcel_id",
    "left_anti"
).count()

# invalid area
invalid_area = metadata_bronze.filter(
    F.col("area_hectares") <= 0
).count()

In [0]:
# Format audit data for table

audit_data = [
    (
        "Mixed date formats",
        invalid_dates,
        builtins.round((invalid_dates / readings_total) * 100, 1),
        "Repair",
        "Needed for correct date analysis"
    ),
    (
        "Invalid NDVI values",
        invalid_ndvi,
        builtins.round((invalid_ndvi / readings_total) * 100, 1),
        "Remove",
        "NDVI should be between -1 and 1"
    ),
    (
        "Duplicate parcel-date records",
        duplicate_count,
        builtins.round((duplicate_count / readings_total) * 100, 1),
        "Remove",
        "One parcel should have one reading per day"
    ),
    (
        "Bad sensor status",
        sensor_issues,
        builtins.round((sensor_issues / readings_total) * 100, 1),
        "Flag",
        "Bad sensor data not used in analysis"
    ),
    (
        "Orphan parcel IDs",
        orphan_parcels,
        builtins.round((orphan_parcels / readings_total) * 100, 1),
        "Remove",
        "No matching metadata found"
    ),
    (
        "Invalid area values",
        invalid_area,
        builtins.round((invalid_area / metadata_total) * 100, 1),
        "Remove",
        "Area should be greater than 0"
    )
]

audit_df = spark.createDataFrame(
    audit_data,
    ["Issue", "Count", "Percentage", "Decision", "Reason"]
)

display(audit_df)

Issue,Count,Percentage,Decision,Reason
Mixed date formats,0,0.0,Repair,Needed for correct date analysis
Invalid NDVI values,104,3.0,Remove,NDVI should be between -1 and 1
Duplicate parcel-date records,8,0.2,Remove,One parcel should have one reading per day
Bad sensor status,340,9.9,Flag,Bad sensor data not used in analysis
Orphan parcel IDs,2,0.1,Remove,No matching metadata found
Invalid area values,0,0.0,Remove,Area should be greater than 0
